# HerBERT-large — ablacja augmentacji, ziarno 44, część B

In [ ]:
# transformers <5.2 — w 5.2 usunięto warmup_ratio z TrainingArguments.
# herbert_large_epochs padł na tym 10.08.2026). Górne ograniczenie utrzymuje
# recepturę identyczną z wcześniejszymi runami tej kampanii.
!pip install -q -U "transformers>=4.44,<5.2" "datasets>=2.20" accelerate 2>/dev/null
import torch, transformers
print(transformers.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
import os, gc, glob, shutil, time, warnings
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.special import expit
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import (f1_score, hamming_loss, jaccard_score, accuracy_score,
                             precision_score, recall_score)
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
warnings.filterwarnings("ignore")
RANDOM_STATE=44; torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
RARE=["strach","zaufanie","smutek"]
FREQUENT=[e for e in EMOTIONS if e not in RARE]
OUT="/kaggle/working"
MODEL_NAME="allegro/herbert-large-cased"; MAX_LEN, EPOCHS, BATCH, LR = 128, 3, 16, 2e-5
RESULT_CSV=f"{OUT}/result.csv"

In [ ]:
def find_csv(n):
    h=glob.glob(f"/kaggle/input/**/{n}",recursive=True)
    if not h: raise FileNotFoundError(f"{n} — dołącz dataset pl-emotion-processed")
    return h[0]

tw_train=pd.read_csv(find_csv("twitteremo_train.csv"))
tw_val=pd.read_csv(find_csv("twitteremo_val.csv"))
tw_test=pd.read_csv(find_csv("twitteremo_test.csv"))
for d in (tw_train,tw_val,tw_test): d["tekst"]=d["tekst"].fillna("")
y_val,y_test=tw_val[EMOTIONS].values,tw_test[EMOTIONS].values

bt=pd.read_csv(find_csv("aug_bt_train.csv")); bt["tekst"]=bt["tekst"].fillna("")
llm=pd.read_csv(find_csv("aug_llm_train.csv")); llm["tekst"]=llm["tekst"].fillna("")
TRAIN_BT=pd.concat([tw_train,bt[["tekst"]+EMOTIONS]],ignore_index=True)
TRAIN_LLM=pd.concat([tw_train,llm[["tekst"]+EMOTIONS]],ignore_index=True)
print("TW",len(tw_train),"| +BT",len(TRAIN_BT),"| +LLM",len(TRAIN_LLM))

In [ ]:
def evaluate(yt,yp):
    return {"f1_macro":f1_score(yt,yp,average="macro",zero_division=0),"f1_micro":f1_score(yt,yp,average="micro",zero_division=0),
            "f1_weighted":f1_score(yt,yp,average="weighted",zero_division=0),"precision_macro":precision_score(yt,yp,average="macro",zero_division=0),
            "recall_macro":recall_score(yt,yp,average="macro",zero_division=0),"hamming_loss":hamming_loss(yt,yp),
            "jaccard_macro":jaccard_score(yt,yp,average="macro",zero_division=0),"subset_accuracy":accuracy_score(yt,yp)}

def find_optimal_thresholds(yt,yp):
    thr=np.full(yt.shape[1],0.5)
    for i in range(yt.shape[1]):
        bf,bt_=0.0,0.5
        for t in np.arange(0.05,0.95,0.01):
            f=f1_score(yt[:,i],(yp[:,i]>=t).astype(int),zero_division=0)
            if f>bf: bf,bt_=f,t
        thr[i]=bt_
    return thr

def f1_macro_ci(yt,yp,n_boot=1000,seed=RANDOM_STATE):
    rng=np.random.default_rng(seed); n=len(yt); base=f1_score(yt,yp,average="macro",zero_division=0)
    b=[f1_score(yt[i],yp[i],average="macro",zero_division=0) for i in (rng.integers(0,n,n) for _ in range(n_boot))]
    lo,hi=np.percentile(b,[2.5,97.5]); return base,lo,hi

In [ ]:
tok=AutoTokenizer.from_pretrained(MODEL_NAME)

class AugTrainer(Trainer):
    """Weighted BCE (pos_weight) and/or weighted sampling of rare rows."""
    def __init__(self,*a,pos_weight=None,sample_weights=None,**k):
        super().__init__(*a,**k); self.pw=pos_weight; self.sample_weights=sample_weights
    def compute_loss(self,model,inputs,return_outputs=False,**kw):
        lab=inputs.pop("labels"); out=model(**inputs)
        loss=F.binary_cross_entropy_with_logits(out.logits.float(),lab.float(),
              pos_weight=self.pw.to(out.logits.device) if self.pw is not None else None)
        return (loss,out) if return_outputs else loss
    def _get_train_sampler(self,*a,**k):
        if self.sample_weights is not None:
            return WeightedRandomSampler(self.sample_weights,len(self.sample_weights),replacement=True)
        return super()._get_train_sampler(*a,**k)

def to_ds(df):
    d=Dataset.from_dict({"text":df["tekst"].tolist(),"labels":df[EMOTIONS].values.astype("float32").tolist()})
    return d.map(lambda b: tok(b["text"],truncation=True,max_length=MAX_LEN),batched=True,remove_columns=["text"])

ds_val,ds_test=to_ds(tw_val),to_ds(tw_test)

def run(name, train_df, use_pos_weight=False, oversample=False):
    done=set()
    if os.path.exists(RESULT_CSV): done=set(pd.read_csv(RESULT_CSV)["warunek"])
    if name in done:
        print(f"== {name}: już policzony, pomijam"); return
    t0=time.time(); print(f"\n=== {name} (n_train={len(train_df)}, pos_weight={use_pos_weight}, oversample={oversample}) ===",flush=True)

    pw=None
    if use_pos_weight:
        # liczony z faktycznego (powiększonego) zbioru — odwrotna częstość tego, co model widzi
        pos=train_df[EMOTIONS].values.sum(0); neg=len(train_df)-pos
        pw=torch.tensor(np.clip(neg/np.maximum(pos,1),1.0,10.0),dtype=torch.float32)
    sw=None
    if oversample:
        sw=(1.0+4.0*(train_df[RARE].sum(1)>0).values.astype(float)).tolist()  # rzadkie x5 częściej

    ds_train=to_ds(train_df)
    model=AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,num_labels=len(EMOTIONS),problem_type="multi_label_classification")
    args=TrainingArguments(output_dir=f"{OUT}/ckpt_{name}",eval_strategy="epoch",save_strategy="epoch",save_total_limit=1,
        load_best_model_at_end=True,metric_for_best_model="f1_macro",greater_is_better=True,per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=32,gradient_accumulation_steps=2,gradient_checkpointing=True,num_train_epochs=EPOCHS,
        learning_rate=LR,warmup_ratio=0.1,weight_decay=0.01,fp16=True,logging_steps=200,report_to="none",seed=RANDOM_STATE)
    cm=lambda p:{"f1_macro":f1_score(p.label_ids.astype(int),(expit(p.predictions)>=0.5).astype(int),average="macro",zero_division=0)}
    trainer=AugTrainer(model=model,args=args,train_dataset=ds_train,eval_dataset=ds_val,
        data_collator=DataCollatorWithPadding(tok),compute_metrics=cm,pos_weight=pw,sample_weights=sw,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()

    p_val=expit(trainer.predict(ds_val).predictions); p_test=expit(trainer.predict(ds_test).predictions)
    thr=find_optimal_thresholds(y_val,p_val); pred=(p_test>=thr).astype(int)
    m=evaluate(y_test,pred); base,lo,hi=f1_macro_ci(y_test,pred)
    m.update({"warunek":name,"n_train":len(train_df),"ci_low":round(lo,3),"ci_high":round(hi,3)})
    for c in RARE:
        i=EMOTIONS.index(c)
        m[f"recall_{c}"]=recall_score(y_test[:,i],pred[:,i],zero_division=0)
        m[f"f1_{c}"]=f1_score(y_test[:,i],pred[:,i],zero_division=0)
    m["f1_frequent_avg"]=float(np.mean([f1_score(y_test[:,EMOTIONS.index(e)],pred[:,EMOTIONS.index(e)],zero_division=0) for e in FREQUENT]))

    hdr=not os.path.exists(RESULT_CSV)
    pd.DataFrame([m]).to_csv(RESULT_CSV,mode="a",header=hdr,index=False)
    np.save(f"{OUT}/proba_test_{name}.npy",p_test)
    np.save(f"{OUT}/proba_val_{name}.npy",p_val)   # do bootstrapu sparowanego
    np.save(f"{OUT}/thr_{name}.npy",thr)
    print(f"   F1-Macro={m['f1_macro']:.3f} [{lo:.3f},{hi:.3f}]  rzadkie: "
          f"strach {m['recall_strach']:.3f} zaufanie {m['recall_zaufanie']:.3f} smutek {m['recall_smutek']:.3f}"
          f"  ({time.time()-t0:.0f}s)",flush=True)
    # punkt kontrolny (4 GB!) jest już niepotrzebny — do dalszych analiz służą
    # zapisane macierze prawdopodobieństw. Bez tego 6 warunków przekracza
    # limit 20 GB katalogu /kaggle/working i kernel pada w połowie.
    shutil.rmtree(args.output_dir, ignore_errors=True)
    del trainer, model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
run("5_llm",             TRAIN_LLM)
run("6_reweight+llm",    TRAIN_LLM, use_pos_weight=True)
run("7_reweight+bt",     TRAIN_BT,  use_pos_weight=True)


In [ ]:
res=pd.read_csv(RESULT_CSV)
print("Odniesienie (HerBERT-base, eksperymenty 10/20): baseline 0,539 | reweight 0,548 |"
      " oversample 0,540 | BT 0,535 | LLM 0,559 | reweight+LLM 0,562\n")
display(res[["warunek","n_train","f1_macro","ci_low","ci_high","f1_micro",
             "recall_strach","f1_frequent_avg"]].round(3))